#**Install necessary libraries**


In [2]:
!pip install pandas
!pip install plotly
!pip install tldextract
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 2.6 MB/s eta 0:00:00


#**Data** Preprocessing


In [3]:
# Importing necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import tldextract

# Loading and preprocessing the dataset
file_path = '/content/py_demo_client_extension_30_20250221075805.csv'
data = pd.read_csv(file_path, skiprows=4)
# Rename columns
data.columns = ['OrgId', 'ParticipantId', 'DeviceId', 'url', 'eventtimeutc',
                'transition', 'title', 'visitId', 'referringVisitId', 'eventtime']

data['eventtimeutc'] = pd.to_datetime(data['eventtimeutc'], errors='coerce')

def extract_domain(url):
    extracted = tldextract.extract(url)
    return f"{extracted.domain}.{extracted.suffix}"

data['domain'] = data['url'].apply(extract_domain)



<ipython-input-3-d977479026cc>:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['eventtimeutc'] = pd.to_datetime(data['eventtimeutc'], errors='coerce')


# Drop duplicates and handle missing data


In [4]:
data.drop_duplicates(inplace=True)
data.dropna(subset=['url', 'eventtimeutc'], inplace=True)
data['hour'] = data['eventtimeutc'].dt.hour
data['date'] = data['eventtimeutc'].dt.date
data['day_of_week'] = data['eventtimeutc'].dt.day_name()

print("Data preprocessing complete!")

Data preprocessing complete!


# Top 10 Most Visited Domains


In [5]:
top_domains = data['domain'].value_counts().head(10).reset_index()
top_domains.columns = ['domain', 'count']

fig = px.bar(top_domains, x='domain', y='count', color='count',
             title="Top 10 Most Visited Domains",
             labels={'domain': 'Domain', 'count': 'Visit Count'})

fig.update_layout(xaxis_title="Domain", yaxis_title="Visit Count",
                  xaxis_tickangle=45)
fig.show()

# Aggregate data to focus only on hours


In [6]:
hourly_aggregate = data.groupby('hour').size().reset_index(name='count')
fig = px.bar(hourly_aggregate, x='hour', y='count',
             title="Browsing Activity by Hour",
             labels={'hour': 'Hour of Day', 'count': 'Visit Count'},
             color='count')

fig.update_layout(xaxis_title="Hour of Day", yaxis_title="Visit Count")
fig.show()

# Activity by Day of the Week


In [7]:
activity_by_day = data['day_of_week'].value_counts().reset_index()
activity_by_day.columns = ['day_of_week', 'count']

fig = px.bar(activity_by_day, x='count', y='day_of_week', color='count',
             orientation='h', title="Browsing Activity by Day of the Week",
             labels={'day_of_week': 'Day', 'count': 'Visit Count'})

fig.update_layout(xaxis_title="Visit Count", yaxis_title="Day of the Week")
fig.show()

# Browsing Activity Over Time


In [8]:
daily_activity = data.groupby('date').size().reset_index(name='count')

fig = px.line(daily_activity, x='date', y='count',
              title="Browsing Activity Over Time",
              labels={'date': 'Date', 'count': 'Number of Visits'})

fig.update_traces(mode='lines+markers')
fig.update_layout(xaxis_title="Date", yaxis_title="Number of Visits")
fig.show()

In [13]:
# Group by hour and referringVisitId for stacked bar chart
stacked_referrals = data.groupby(['hour', 'referringVisitId']).size().reset_index(name='count')

# Create stacked bar chart
fig = px.bar(stacked_referrals, x='hour', y='count', color='referringVisitId',
             title="Referring Visits Grouped by Hour",
             labels={'hour': 'Hour of Day', 'count': 'Visit Count', 'referringVisitId': 'Referring Visit ID'})

fig.update_layout(xaxis_title="Hour of Day", yaxis_title="Visit Count")
fig.show()

In [10]:
# Transition Types and Visit Counts
transition_activity = data.groupby('transition').size().reset_index(name='count')

# Plotly Pie Chart for Transition Types
fig = px.pie(transition_activity, names='transition', values='count',
             title="Distribution of Transition Types",
             labels={'transition': 'Transition Type', 'count': 'Visit Count'})

fig.update_traces(textinfo='percent+label')
fig.show()